In [ ]:
pip install -q kaggle

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"abhi2op","key":"adba602b7bd555cd31bb4316c6b46cc5"}'}

In [ ]:
#step 3: Move kaggle.json to the Correct location
import os
import shutil
#Create the kaggle folder
os.makedirs('/root/.kaggle',exist_ok=True)
#Move kaggle.json
shutil.move('/content/kaggle.json','/root/.kaggle')

'/root/.kaggle/kaggle.json'

In [ ]:
print("Kaggle API configured successfully")

Kaggle API configured successfully


In [ ]:
#download the cat vs  dog dataset

!kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset

Dataset URL: https://www.kaggle.com/datasets/shaunthesheep/microsoft-catsvsdogs-dataset
License(s): other
100% 788M/788M [00:04<00:00, 182MB/s]



In [ ]:
#extract the dataset
import zipfile
with zipfile.ZipFile("microsoft-catsvsdogs-dataset.zip","r") as zip_ref:
  zip_ref.extractall("dataset")

In [ ]:
#check dataset
print(os.listdir("dataset"))

print(os.listdir("dataset/PetImages"))

['PetImages', 'MSR-LA - 3467.docx', 'readme[1].txt']
['Dog', 'Cat']


In [ ]:
print("Cats:", len(os.listdir("dataset/PetImages/Cat")))
print("Dogs:", len(os.listdir("dataset/PetImages/Dog")))

Cats: 12501
Dogs: 12501


In [ ]:
#split the dataset
import os
import shutil
import random

random.seed(42)

cat_source="dataset/PetImages/Cat"
dog_source="dataset/PetImages/Dog"

train_cat="dataset/train/cat"
train_dog="dataset/train/dog"

val_cat="dataset/val/cat"
val_dog="dataset/val/dog"

In [ ]:
for folder in [train_cat,train_dog,val_cat,val_dog]:
  os.makedirs(folder,exist_ok=True)

In [ ]:
def split_dataset(source,train_dest,val_dest):
  images=[]

  for file in os.llistdir(sourrce):
    path=os.ath.join(source, file)

    try:
      if os.path.getsize(path)>0:
        images.append(file)
    except:
      pass

    random.shuffle(images)

    split=int(len(images)*0.8)

    train=images[:split]

    val=images[split:]

    for img in train:
      shutil.copyfile(os.path.join(source,img),os.path.join(train_dest,img))

     # Move validation images
    for img in val_images:
      shutil.copy(os.path.join(source, img), os.path.join(val_dest, img))

    print(f"Dataset split: {len(train_images)} training images, {len(val_images)} validation images.")




In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
#load dataset
IMAGE_SIZE=(150,150)
BATCH_SIZE=32

train_datagen=ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
)

In [ ]:
validation_datagen=ImageDataGenerator(
    rescale=1./255
)

In [ ]:
train_generator=train_datagen.flow_from_directory(
    '/dataset/train',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

FileNotFoundError: [Errno 2] No such file or directory: '/dataset/train'

In [ ]:
validation_geneeator=train_datagen.flow_from_directory(
    '/dataset/val',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

In [ ]:
model=tf.keras.Sequential([
    tf.keras.layers.Input(shape=(150,150,3)),
    tf.keras.layers.Conv2D(32,(3,3),activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(64,(3,3),activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(128,(3,3),activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(128,(3,3),activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(512,activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1,activation="sigmoid")
])

In [ ]:
model.summary()

In [ ]:
#compile model

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

In [ ]:
#train model

early_stop=tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)


In [ ]:
history=model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=10,
    callbacks=[early_stop]
)

In [ ]:
#check accuracy

loss,accuracy=model.evaluate(validation_generator)
print("Validation Loss:",loss)
print("Validation Accuracy:",accuracy)

In [ ]:
#plot accuracy

plt.figure(figsize=(10,5))

plt.plot(history.history['accuracy'],label="Training Accuracy")
plt.plot(history.history['val_accuracy'],label="Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
#save model
model.save('dog_cat_model.keras')
print('Model Saved')

In [ ]:
#upload new image
from google.colab import files
uploaded=file.upload()

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

In [ ]:
img_path=list(uploaded.keys())[0]

img=image.load_img(img_path,target_size=(150,150))

plt.imshow(img)
plt.axis("off")
plt.show()

img_array=image.img_to_array(img)

img_array=img_array/255.0

img_array=np.expand_dims(img_array,axis=0)

prediction=model.predict(img_array)

score=prediction[0][0]

if score>=0.5:
  print("Prediction : Dog")
  print("Confidence : ", round(score*100,2)"%")
else:
  print("Prediction : Cat")
  print("Confidence : ",round((1-core)*100,2),"%")

In [ ]:
#convert keras model to tensorflow lite

import tensorflow as tf

#load trained model

model=tf.keras.models.load_model('dog_cat_model.keras')

#convert model to tensorflow lite

converter=tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model=converter.convert()

In [ ]:
#save the model
with open("cat_dog_model.tflite","wb") as f:
  f.write(tflite_model)

print("TFLite model created successfully!")


In [ ]:
from google.colab import files
files.download("cat_dog_model.tflite")
